<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/StableDiffusionChapter19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install diffusers
!pip install transformers scipy ftfy accelerate ipywidgets

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "stablediffusionapi/deliberate-v2"
text2img_pipe = StableDiffusionPipeline.from_pretrained(
    model_id
    , torch_dtype = torch.float16
)

In [ ]:
gen_meta = {
    "model_id": model_id
    , "prompt": "high resolution, a photograph of an astronaut riding a horse"
    , "seed": 123
    , "inference_steps": 30
    , "height": 512
    , "width": 768
    , "guidance_scale": 7.5
}

In [ ]:
text2img_pipe.to("cuda:0")
input_image = text2img_pipe(
    prompt            = gen_meta["prompt"]
    , generator       = torch.Generator("cuda:0").manual_seed(gen_meta["seed"])
    , guidance_scale  = gen_meta["guidance_scale"]
    , height          = gen_meta["height"]
    , width           = gen_meta["width"]
).images[0]
text2img_pipe.to("cpu")
torch.cuda.empty_cache()
input_image

In [ ]:
!pip install pillow

In [ ]:
from PIL import Image
from PIL import PngImagePlugin
import json

# Open the original image
image = Image.open("input_image.png")

# Define the metadata you want to add
metadata = PngImagePlugin.PngInfo()
gen_meta_str = json.dumps(gen_meta)
metadata.add_text("my_sd_gen_meta", gen_meta_str)

# Save the image with the added metadata
image.save("output_image_with_metadata.png", "PNG", pnginfo=metadata)

In [ ]:
from PIL import Image
from PIL import PngImagePlugin
import json

# Open the original image
image = input_image#Image.open("input_image.png")

# Define the metadata you want to add
metadata = PngImagePlugin.PngInfo()
gen_meta_str = json.dumps(gen_meta)
metadata.add_text("my_sd_gen_meta", gen_meta_str)

# add a copy right json object
copyright_meta = {
    "author":"Andrew Zhu"
    ,"license":"free use"
}
copyright_meta_str = json.dumps(copyright_meta)
metadata.add_text("copy_right", copyright_meta_str)

# Save the image with the added metadata
image.save("output_image_with_metadata.png", "PNG", pnginfo=metadata)

In [ ]:
from PIL import Image
image = Image.open("output_image_with_metadata.png")

metadata = image.info

# print the meta
for key, value in metadata.items():
    print(f"{key}: {value}")